# معمل التجربة: إتقان سلاسل LCEL في الـ RAG

لغة التعبير في لانج تشين (LangChain Expression Language أو LCEL) بتخليك تركب مكونات الـ RAG بسهولة زي المكعبات باستخدام علامة الـ Pipe (`|`).

في المعمل ده هنتعلم:
1. إزاي نربط المسترجع مع المنسق مع البرومبت مع الموديل في سطر واحد.
2. إزاي نخلي الإجابة تطلع كلمة كلمة بنظام البث المباشر (Streaming).

### 1. تجهيز الفهرس والمسترجع
يلا نجهز فهرس خفيف في الذاكرة:

In [1]:
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv()
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

data = [
    Document(page_content="الشبكات العصبية الالتفافية (CNN) مناسبة جداً للتعامل مع الصور والرؤية الحاسوبية."),
    Document(page_content="الشبكات العصبية المتكررة (RNN) تستخدم لمعالجة البيانات المتسلسلة مثل السلاسل الزمنية."),
    Document(page_content="نماذج المحولات (Transformers) تفوقت على الشبكات المتكررة بفضل آلية الانتباه الذاتي."),
]

vector_store = FAISS.from_documents(data, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
print("المسترجع جاهز!")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_4976\536957446.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

المسترجع جاهز!


### 2. بناء مسار LCEL وتجربة البث المباشر (Streaming)
لاحظ بساطة الكود:

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)

prompt = ChatPromptTemplate.from_template("""أجب باختصار بالاعتماد على السياق:
السياق: {context}
السؤال: {question}
الإجابة:""")

def format_docs(docs):
    return "\n".join(d.page_content for d in docs)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("جاري استقبال الإجابة كلمة بكلمة (Streaming):\n")
for chunk in chain.stream("ما هي أفضل شبكة لمعالجة الصور؟"):
    print(chunk, end="", flush=True)
print("\n")

جاري استقبال الإجابة كلمة بكلمة (Streaming):



الش

ب

كة

 العص

بية

... (2 outputs left out)

افية

 (

CNN

).

### 3. جرب بنفسك!
تقدر تعدل السؤال في دالة الـ stream وتشوف سرعة التوليد بنفسك.